# Normalization H1-H4

Run all cells. Existing model transformations are preserved; citations are not votes.
The CSV reports evidence limitations, correct bibliographic details, PDF paths and PDF page numbers.
Every cited full text is validated by identity, evidence-page text and SHA256. No network access is needed.
Only the two existing empatica/eyetracker output CSVs are written.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
import os
import hashlib
import pymupdf
import tempfile

# 1. Config
ROOT = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final")
DATA = ROOT / "output/merged_all_11participants_8min.csv"  # same input as H1_H3_phaseavg.R / H4_timeseries.R
CORE_PHASES = ["BikeU", "BikeG", "WalkU", "WalkG"]  # x <= 0 counts use the analysed core phases only
PAPERS = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Github/Experiment papers")
OUT_DIR = ROOT / "output/statistical_analysis/Normalization & Parameter Screening/Normalization"

# transformation_in_models labels; HRV and EDA tonic are ln(x), never ln(1 + x). paper3_log_outcomes() in scripts/statistical_analysis/library/Paper3_common.R accepts only these.
STEP5_TEXT = {
    "No transformation": "No transformation",
    "Natural log": "Natural log",
}
# Physiological outcomes fitted in H4 Step 5 (h4_study_spec$physiological_outcome_names in scripts/H4_timeseries.R).
# empatica__eda_scl_usiemens is not a Step 5 outcome.
NOT_IN_STEP5 = {"empatica__eda_scl_usiemens"}
NORMALIZATION_PREFIXES = ("empatica", "eyetracker")

def in_normalization_scope(column: str) -> bool:
    return column.startswith(NORMALIZATION_PREFIXES)

REFERENCES = {'Romine2020': {'file': 'LMM_transform_evidence/Normalization - Romine et al 2020 - Wearable '
                        'cognitive load tracker.pdf',
                'citation': 'Romine et al. (2020). Using Machine Learning to Train a Wearable '
                            "Device for Measuring Students' Cognitive Load during Problem-Solving "
                            'Activities Based on Electrodermal Activity, Body Temperature, and '
                            'Heart Rate: Development of a Cognitive Load Tracker for Both Personal '
                            'and Classroom Use. Sensors, 20(17), 4833.',
                'doi': '10.3390/s20174833',
                'identity': 'romine',
                'url': 'https://mdpi-res.com/d_attachment/sensors/sensors-20-04833/article_deploy/sensors-20-04833.pdf',
                'pages': '5',
                'anchor': 'Transformation of Physiological Features',
                'evidence': 'Section 3.1.2 scales physiological features within participants. '
                            'Context for HR/temperature measurement and scaling; does not '
                            'establish that native-scale outcomes are optimal in our mixed models.',
                'sha256': 'd7dce3c105c7fdb67fac32e61cf820f501135c95cfe063496e7e5cd9aa418f1d'},
 'Beggiato2018': {'file': 'LMM_transform_evidence/Normalization - Beggiato et al 2018 - Smartbands '
                          'pupillometry and body motion.pdf',
                  'citation': 'Beggiato, Hartwich & Krems (2018). Using Smartbands, Pupillometry '
                              'and Body Motion to Detect Discomfort in Automated Driving. '
                              'Frontiers in Human Neuroscience, 12, 338.',
                  'doi': '10.3389/fnhum.2018.00338',
                  'identity': 'beggiato',
                  'url': 'https://www.frontiersin.org/journals/human-neuroscience/articles/10.3389/fnhum.2018.00338/pdf',
                  'pages': '7-8',
                  'anchor': 'z-transformation',
                  'evidence': 'Pupil values are z-standardized and adjusted for luminance; SCL is '
                              'standardized. Measurement/scaling context, not an endorsement of '
                              'untransformed pupil or EDA outcomes.',
                  'sha256': '1700794697b51c6a061e615e896dd31ac618fd0ed78b506dde6fc11de98419ef'},
 'Madaniyazi2016': {'file': 'Air temp - Outdoor temperature heart rate and blood pressure in '
                            'Chinese adults.pdf',
                    'citation': 'Madaniyazi et al. (2016). Outdoor Temperature, Heart Rate and '
                                'Blood Pressure in Chinese Adults: Effect Modification by '
                                'Individual Characteristics. Scientific Reports, 6, 21003.',
                    'doi': '10.1038/srep21003',
                    'identity': 'madaniyazi',
                    'pages': '1',
                    'anchor': 'heart rate',
                    'evidence': 'Temperature-heart-rate association and HR measurement context; '
                                'not proof that the current repeated-measures residuals require no '
                                'transformation.',
                    'sha256': 'bf8bd49ef98d0d89a7530135ac73ef610bdcebf50c7f897e832174f645d6670e'},
 'Liu2019': {'file': 'LMM_transform_evidence/Personal_thermal_comfort_wearable_sensors_2018.pdf',
             'citation': 'Liu, Schiavon, Das, Jin & Spanos (2019). Personal thermal comfort models '
                         'with wearable sensors. Building and Environment, 162, 106281.',
             'doi': '10.1016/j.buildenv.2019.106281',
             'identity': 'shichao',
             'pages': '1-2',
             'anchor': 'skin temperature',
             'evidence': 'Skin temperature/heart rate are features in personal thermal-comfort '
                         'prediction. Predictor use in machine learning does not validate our '
                         'native-scale mixed-model outcomes. Legacy PDF filename has the wrong '
                         'year.',
             'sha256': '93296d8b225c203b8e9f6a5bac5b20c1ea467091eddcbb198c707d70fa512540'},
 'Mansi2022': {'file': 'Empatica - A novel methodology for human thermal comfort decoding via '
                       'physiological signals.pdf',
               'citation': 'Mansi, Pigliautile, Arnesano & Pisello (2022). A novel methodology for '
                           'human thermal comfort decoding via physiological signals measurement '
                           'and analysis. Building and Environment, 222, 109385.',
               'doi': '10.1016/j.buildenv.2022.109385',
               'identity': 'mansi',
               'pages': '1',
               'anchor': 'skin temperature',
               'evidence': 'Thermal-comfort physiological measurement context including skin '
                           'temperature; not direct validation of our outcome transformation.',
               'sha256': 'a064f1aa07ba846c67361455b4006376777efbe9966bc08e0ab26d7d67442464'},
 'Boucsein2012': {'file': 'LMM_transform_evidence/EDA_recommendations_BIOPAC_Psychophysiology_2012.pdf',
                  'citation': 'Boucsein et al. (2012). Publication recommendations for '
                              'electrodermal measurements. Psychophysiology, 49(8), 1017-1034.',
                  'doi': '10.1111/j.1469-8986.2012.01384.x',
                  'identity': 'boucsein',
                  'pages': '9-10',
                  'anchor': 'log scl',
                  'evidence': 'Section 3.1: SCR amplitudes are often log- or square-root-transformed. '
                              'Section 3.2: "SCL is expressed in microsiemens units but is often '
                              'transformed to log SCL in order to normalize the data." Methodological '
                              'guidance, not model-specific validation.',
                  'sha256': 'f9f661aa62e41733c35253c2eb038ec3a73cc44f4e23a66d307f14d75604d2d6'},
 'Knejzlikova2021': {'file': 'LMM_transform_evidence/EDA - Vocks et al 2021 - Electrodermal '
                             'Response to Mirror Exposure in Restrictive Anorexia.pdf',
                     'citation': 'Knejzlikova et al. (2021). Electrodermal Response to Mirror '
                                 'Exposure in Relation to Subjective Emotional Responses, '
                                 'Emotional Competences and Affectivity in Adolescent Girls With '
                                 'Restrictive Anorexia and Healthy Controls. Frontiers in '
                                 'Psychology, 12, 673597.',
                     'doi': '10.3389/fpsyg.2021.673597',
                     'identity': '673597',
                     'pages': '5',
                     'anchor': 'transformed using',
                     'evidence': 'Statistical Analysis: SCR uses ln(0.00001+x), SCL uses square '
                                 'root. Example of transformed EDA, not support for our '
                                 'native-scale choice. Vocks is an incorrect legacy filename '
                                 'attribution.',
                     'sha256': '1c54ea36ac5a4d28de2bf7c07b731a08aae6ca75ebabfb3d500443ca7457874d'},
 'Browning2020': {'file': 'LMM_transform_evidence/EDA - Browning et al 2020 - Can simulated nature '
                          'support mental health 360-degree nature videos.pdf',
                  'citation': 'Browning, Mimnaugh, van Riper, Laurent & LaValle (2020). Can '
                              'Simulated Nature Support Mental Health? Comparing Short, '
                              'Single-Doses of 360-Degree Nature Videos in Virtual Reality With '
                              'the Outdoors. Frontiers in Psychology, 10, 2667.',
                  'doi': '10.3389/fpsyg.2019.02667',
                  'identity': 'browning',
                  'url': 'https://www.frontiersin.org/journals/psychology/articles/10.3389/fpsyg.2019.02667/pdf',
                  'pages': '6',
                  'anchor': 'log-transformed to correct skew',
                  'evidence': 'Statistical Analyses: repeated SCL means for twelve 30 s intervals '
                              'were "natural log-transformed to correct skew" and modeled with '
                              'multilevel growth curves. Empirical precedent for ln SCL in '
                              'repeated-measures mixed models.',
                  'sha256': 'dabe1b6948ef5ad05fb7b6a39b4a6ba10ef78e0c66784fc049f03b7583952608'},
 'Jin2023': {'file': 'LMM_transform_evidence/EDA - Jin et al 2023 - Stress recovery effects of '
                     'viewing simulated urban parks.pdf',
             'citation': 'Jin, Wang, Liu, Han, Qi & Wang (2023). Stress Recovery Effects of '
                         'Viewing Simulated Urban Parks: Landscape Types, Depressive Symptoms, and '
                         'Gender Differences. Land, 12(1), 22. Published online 21 December 2022; '
                         'issue year 2023.',
             'doi': '10.3390/land12010022',
             'identity': 'ziliang jin',
             'url': 'https://mdpi-res.com/d_attachment/land/land-12-00022/article_deploy/land-12-00022.pdf',
             'pages': '7',
             'anchor': 'right-skewed',
             'evidence': 'Data analysis: "Because the SCL data were right-skewed, we used a '
                         'logarithmic transformation for original measures of the SCL to reduce '
                         'kurtosis." Empirical precedent for log SCL.',
             'sha256': '73d31ba37748cf6a45832bfb349f98b47dfb543198b26fe1a635f27d802fdd79'},
 'Bulteel2014': {'file': 'LMM_transform_evidence/EDA - Bulteel et al 2014 - DeCon emotional '
                         'concordance in multivariate time series.pdf',
                 'citation': 'Bulteel, Ceulemans, Thompson, Waugh, Gotlib, Tuerlinckx & Kuppens '
                             '(2014). DeCon: A tool to detect emotional concordance in '
                             'multivariate time series data of emotional responding. Biological '
                             'Psychology, 98, 29-42. Published online 9 November 2013; issue year '
                             '2014.',
                 'doi': '10.1016/j.biopsycho.2013.10.011',
                 'identity': 'bulteel',
                 'url': 'https://www.ppw.kuleuven.be/okp/_pdf/Bulteel2014DATDE.pdf',
                 'pages': '6',
                 'anchor': 'were log-transformed',
                 'evidence': 'Data preprocessing: "IBI and SCL values were log-transformed" before '
                             'DeCon change detection and auxiliary (multilevel) regression. '
                             'Empirical precedent for log SCL.',
                 'sha256': 'ef6c436486b36c677066e60908c97df62dbb694732e93242177518a2c50b48cf'},
 'Liu2015': {'file': 'LMM_transform_evidence/EDA - Liu et al 2015 - Sleep deprived and sweating it '
                     'out skin conductance reactivity.pdf',
             'citation': 'Liu, Verhulst, Massar & Chee (2015). Sleep Deprived and Sweating It Out: '
                         'The Effects of Total Sleep Deprivation on Skin Conductance Reactivity to '
                         'Psychosocial Stress. Sleep, 38(1), 155-159.',
             'doi': '10.5665/sleep.4346',
             'identity': 'verhulst',
             'url': 'https://sleepcognition-lab.org/wp-content/uploads/2022/01/2015_10_Liuetal.2015.pdf',
             'pages': '3',
             'anchor': 'scl + 1',
             'evidence': 'Data Analyses: SCL means "were log-transformed (ln[SCL + 1])". Empirical '
                         'precedent for log SCL; this project uses ln(x) because tonic EDA has no '
                         'nonpositive values.',
             'sha256': '787cdbe4e68a35507da25d2c2f72792d5df378f1fad327a988415f2578ec156d'},
 'Laborde2017': {'file': 'LMM_transform_evidence/HRV - Laborde Mosley Thayer 2017 - Heart Rate '
                         'Variability and Cardiac Vagal Tone in Psychophysiological Research '
                         'Recommendations.pdf',
                 'citation': 'Laborde, Mosley & Thayer (2017). Heart Rate Variability and Cardiac '
                             'Vagal Tone in Psychophysiological Research - Recommendations for '
                             'Experiment Planning, Data Analysis, and Data Reporting. Frontiers in '
                             'Psychology, 8, 213.',
                 'doi': '10.3389/fpsyg.2017.00213',
                 'identity': 'laborde',
                 'pages': '12',
                 'anchor': 'natural logarithm',
                 'evidence': 'Normality of HRV Parameters: natural-log RMSSD and absolute powers '
                             'are examples for non-normal HRV. Conditional guidance, not a mandate '
                             'to log every HRV outcome.',
                 'sha256': 'bee025e3a8868b7afd2fbaaf22ddd2ea11c13ac47df600cbe84056f584d53971'},
 'Shaffer2017': {'file': 'Empatica - Duration - An Overview of Heart Rate Variability Metrics and '
                         'Norms.pdf',
                 'citation': 'Shaffer & Ginsberg (2017). An Overview of Heart Rate Variability '
                             'Metrics and Norms. Frontiers in Public Health, 5, 258.',
                 'doi': '10.3389/fpubh.2017.00258',
                 'identity': 'shaffer',
                 'pages': '2',
                 'anchor': 'original units',
                 'evidence': 'Time-domain values may be expressed in original or natural-log units '
                             'to improve distributional shape. Definitions and conditional '
                             'transformation rationale, not model-specific validation.',
                 'sha256': '2977f58913d4e301962bad1a132d823a1e42fb19ce6f909e607ddc81e360b881'},
 'TaskForce1996': {'file': 'Empatica - HRV_task_force_team_europe_1996-1.pdf',
                   'citation': 'Task Force of the ESC and NASPE (1996). Heart rate variability: '
                               'Standards of measurement, physiological interpretation, and '
                               'clinical use. European Heart Journal, 17(3), 354-381.',
                   'doi': '10.1093/oxfordjournals.eurheartj.a014868',
                   'identity': '354',
                   'pages': '6',
                   'anchor': 'normalized units',
                   'evidence': 'Defines absolute powers, normalized LF/HF units and their '
                               'interpretation. Normalized units are a physiological quantity, not '
                               'proof of normality or a prohibition on statistical '
                               'transformations.',
                   'sha256': '9475bb5b6104d4334d1ad77bd38c0ff31b77448eac50784085bf90f3a5a1d218'},
 'Mahanama2022': {'file': 'Eyetracker - Mahanama et al 2021 - Eye movement and pupil measures '
                          'review.pdf',
                  'citation': 'Mahanama et al. (2022). Eye Movement and Pupil Measures: A Review. '
                              'Frontiers in Computer Science, 3, 733531.',
                  'doi': '10.3389/fcomp.2021.733531',
                  'identity': 'mahanama',
                  'pages': '1-4',
                  'anchor': 'velocity',
                  'evidence': 'Definitions of eye movements, blinking and pupil measures. Does not '
                              'prescribe no transformation for every derived metric. Published 11 '
                              'January 2022 in volume 3 (2021); legacy filename retains 2021.',
                  'sha256': '17a7577ef79c776161723167fd6649396bc310c09912ed8f9cb9cc298763f5be'},
 'Evans2012': {'file': 'Eyetracker - Evans et al 2012 - Collecting and analyzing eye-tracking data '
                       'in outdoor environments.pdf',
               'citation': 'Evans, Jacobs, Tarduno & Pelz (2012). Collecting and Analyzing '
                           'Eye-Tracking Data in Outdoor Environments. Journal of Eye Movement '
                           'Research, 5(2), 6, 1-19.',
               'doi': '10.16910/jemr.5.2.6',
               'identity': 'evans',
               'pages': '12',
               'anchor': 'dispersion',
               'evidence': 'Outdoor calibration and fixation-detection methods, including '
                           'dispersion/duration thresholds. Methods context; not evidence that '
                           'statistical transformations are unnecessary.',
               'sha256': 'd69aae4788e34487dc332f853fb8e37b40a8956150e4f2b2393099da78570543'},
 'Kchour2025': {'file': 'LMM_transform_evidence/Normalization - Kchour et al 2025 - Cyclists '
                        'visual behavior systematic review.pdf',
                'citation': "Kchour, Cafiso & Pappalardo (2025). Understanding Cyclists' Visual "
                            'Behavior Using Eye-Tracking Technology: A Systematic Review. Sensors, '
                            '25(1), 22. Published online 24 December 2024; issue year 2025.',
                'doi': '10.3390/s25010022',
                'identity': 'kchour',
                'url': 'https://mdpi-res.com/d_attachment/sensors/sensors-25-00022/article_deploy/sensors-25-00022.pdf',
                'pages': '1',
                'anchor': 'visual behavior',
                'evidence': 'Cycling eye-tracking context and visual-attention measures. Review '
                            'does not validate all derived measures or prescribe our '
                            'no-transformation policy.',
                'sha256': '4fafcb4d123abcd314cc310ff7639b39a84991b031d84efc7e2817be019c17e5'}}

USED = {'Heart rate': [('No transformation', ['Romine2020', 'Beggiato2018', 'Madaniyazi2016'])],
 'Skin temperature': [('No transformation', ['Romine2020', 'Liu2019', 'Mansi2022'])],
 'Skin conductance level': [('No transformation', ['Boucsein2012', 'Knejzlikova2021'])],
 'Skin conductance, tonic': [('Natural log', ['Boucsein2012', 'Browning2020', 'Jin2023', 'Bulteel2014', 'Liu2015'])],
 'Skin conductance, phasic': [('No transformation', ['Boucsein2012', 'Knejzlikova2021'])],
 'SDNN': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'RMSSD': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'LF power': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'HF power': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'VLF power': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'Total power': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'LF/HF ratio': [('Natural log', ['Laborde2017', 'Shaffer2017'])],
 'LF normalized': [('No transformation', ['TaskForce1996', 'Laborde2017'])],
 'HF normalized': [('No transformation', ['TaskForce1996', 'Laborde2017'])],
 'Pupil diameter': [('No transformation', ['Mahanama2022', 'Beggiato2018'])],
 'Pupil change rate': [('No transformation', ['Mahanama2022'])],
 'Eyelid aperture': [('No transformation', ['Mahanama2022'])],
 'Blink duration': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Fixation duration': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Saccade duration': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Fixation rate': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Saccade amplitude': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Saccade peak velocity': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Gaze velocity': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Gaze dispersion': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Distance from center': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])],
 'Gaze centrality': [('No transformation', ['Mahanama2022', 'Evans2012', 'Kchour2025'])]}

COL = {
    "Heart rate": "empatica_heart_rate",
    "Skin temperature": "empatica_skin_temperature",
    "Skin conductance level": "empatica__eda_scl_usiemens",
    "Skin conductance, tonic": "empatica_eda_tonic",
    "Skin conductance, phasic": "empatica_eda_phasic",
    "SDNN": "empatica_hrv_td_sdnn",
    "RMSSD": "empatica_hrv_td_rmssd",
    "LF power": "empatica_hrv_fd_lf",
    "HF power": "empatica_hrv_fd_hf",
    "VLF power": "empatica_hrv_fd_vlf",
    "Total power": "empatica_hrv_fd_total_power",
    "LF/HF ratio": "empatica_hrv_fd_lf_hf_ratio",
    "LF normalized": "empatica_hrv_fd_lfnu",
    "HF normalized": "empatica_hrv_fd_hfnu",
    "Pupil diameter": "eyetracker_pupil_diameter_avg",
    "Pupil change rate": "eyetracker_pupil_change_rate",
    "Eyelid aperture": "eyetracker_eyelid_aperture_avg",
    "Blink duration": "eyetracker_blink_duration_s",
    "Fixation duration": "eyetracker_fixation_duration_s",
    "Saccade duration": "eyetracker_saccade_duration_s",
    "Fixation rate": "eyetracker_fixation_rate",
    "Saccade amplitude": "eyetracker_saccade_amplitude",
    "Saccade peak velocity": "eyetracker_saccade_peak_velocity",
    "Gaze velocity": "eyetracker_gaze_velocity",
    "Gaze dispersion": "eyetracker_gaze_dispersion",
    "Distance from center": "eyetracker_distance_from_center",
    "Gaze centrality": "eyetracker_gaze_centrality",
}

UNIT = {
    "empatica_heart_rate": "bpm", "empatica_skin_temperature": "°C", "empatica__eda_scl_usiemens": "microsiemens",
    "empatica_eda_tonic": "microsiemens", "empatica_eda_phasic": "microsiemens", "empatica_hrv_td_sdnn": "ms",
    "empatica_hrv_td_rmssd": "ms", "empatica_hrv_fd_lf": "ms2", "empatica_hrv_fd_hf": "ms2", "empatica_hrv_fd_vlf": "ms2",
    "empatica_hrv_fd_total_power": "ms2", "empatica_hrv_fd_lf_hf_ratio": "ratio", "empatica_hrv_fd_lfnu": "%",
    "empatica_hrv_fd_hfnu": "%", "eyetracker_pupil_diameter_avg": "mm", "eyetracker_pupil_change_rate": "mm/s",
    "eyetracker_eyelid_aperture_avg": "mm", "eyetracker_blink_duration_s": "s", "eyetracker_fixation_duration_s": "s",
    "eyetracker_saccade_duration_s": "s", "eyetracker_fixation_rate": "per second", "eyetracker_saccade_amplitude": "degrees",
    "eyetracker_saccade_peak_velocity": "degrees/s", "eyetracker_gaze_velocity": "degrees/s",
    "eyetracker_gaze_dispersion": "degrees", "eyetracker_distance_from_center": "pixels", "eyetracker_gaze_centrality": "proportion",
}
assert set(USED) == set(COL), set(USED) ^ set(COL)
assert set(COL.values()) == set(UNIT), set(COL.values()) ^ set(UNIT)
assert {t for opts in USED.values() for t, _ in opts} <= set(STEP5_TEXT)

# Bilateral means used by H4, in addition to every source column.
AVERAGES = {
    **{f"atmo_pm{size}_avg": [f"atmo_lys_atmotube_{side}__atmotube_pm{size}" for side in ["left", "right"]]
       for size in ["1", "2.5", "10"]},
    **{f"lys_{measure}_avg": [f"atmo_lys_LYS{sensor}__lys_{measure}" for sensor in [1, 2]]
       for measure in ["lux", "medi", "kelvin"]},
}
SOUND_PROPORTION_PREFIXES = ("freq_prop_", "octave_proportion_")
QC_FIELDS = {"pct_complete", "gvi_match_dist_m", "SVF_Frames_In_Window", "segment", "end_time_s", "phase_start_offset_s"}
METADATA_FIELDS = {"ParticipantID", "PhaseID", "Datetime", "Date"}



print("config ok: %d physio parameters mapped; citations describe evidence, not votes" % len(USED))


config ok: 27 physio parameters mapped; citations describe evidence, not votes


In [2]:
# 2. Load
raw = pd.read_csv(DATA, low_memory=False)
missing_cols = [c for c in COL.values() if c not in raw.columns]
assert not missing_cols, missing_cols

paper_paths = {}
for key, reference in REFERENCES.items():
    path = PAPERS / reference["file"]
    payload = path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != reference["sha256"]:
        raise ValueError(f"Citation PDF changed and needs re-verification: {path}")
    with pymupdf.open(stream=payload, filetype="pdf") as document:
        pages = [re.sub(r"\s+", " ", page.get_text()).lower() for page in document]
    if len(pages) < 2 or sum(map(len, pages)) < 10000 or reference["identity"] not in " ".join(pages):
        raise ValueError(f"Citation is not the verified full text: {path}")
    start, _, end = reference["pages"].partition("-")
    evidence_pages = " ".join(pages[int(start)-1:int(end or start)])
    assert reference["anchor"].lower() in evidence_pages, key
    paper_paths[key] = path
core_values = raw.loc[raw.PhaseID.isin(CORE_PHASES), list(COL.values())].apply(pd.to_numeric, errors="coerce")
nonpositive_counts = core_values.le(0).sum()

normalization_columns = sorted(c for c in raw.columns if in_normalization_scope(c))
core = raw.loc[raw.PhaseID.isin(CORE_PHASES)].copy()
print("loaded %d rows (%d core phases), %d literature PDFs resolved, %d normalization columns" % (len(raw), len(core), len(paper_paths), len(normalization_columns)))


loaded 5077 rows (2156 core phases), 13 literature PDFs resolved, 41 normalization columns


In [3]:
# 3. Define
def step5_text(col_name, transformation):
    return "Not applicable" if col_name in NOT_IN_STEP5 else STEP5_TEXT[transformation]

def check_log_domain(col_name, transformation):
    return int(nonpositive_counts[col_name]) if transformation == "Natural log" else 0


def make_normalization_tables():
    numbers = {key: f"[{i}]" for i, key in enumerate(REFERENCES, 1)}
    rows, evidence = [], []
    for parameter, options in USED.items():
        transformation, keys = options[0]
        column = COL[parameter]
        if parameter == "Skin conductance, tonic":
            status = "Log SCL guidance and empirical precedent; current-model choice not validated"
            rationale = f"Tonic EDA is a positive, right-skewed level (core-phase skewness {core_values[column].skew():.1f}, {int(nonpositive_counts[column])} nonpositive values). Natural log follows Boucsein et al. (2012) log SCL guidance and four empirical studies that log-transformed SCL. Residual diagnostics remain necessary."
        elif "conductance" in parameter.lower():
            status = "Alternative transformations documented; native-scale choice not validated"
            rationale = f"EDA guidance discusses log/square-root SCR and log SCL. This series has {int(nonpositive_counts[column])} nonpositive core-phase values, so ln(x) is undefined; continuous decomposed phasic EDA is a signed signal, not event-level SCR amplitude. Native scale retained pending outcome-specific model diagnostics."
        elif transformation == "Natural log":
            status = "Conditional/general HRV rationale; current-model choice not validated"
            rationale = "Natural-log HRV is supported for suitable non-normal measures, with explicit examples for RMSSD and absolute powers. This does not demonstrate that each measure or current model needs a log. Residual diagnostics and model context remain necessary."
        elif parameter in {"LF normalized", "HF normalized"}:
            status = "Quantity definition; no direct endorsement of transformation choice"
            rationale = "Normalized LF/HF are physiological units, not a guarantee of Gaussian residuals. Native-scale analysis is a declared project decision."
        else:
            status = "Measurement/method context only; no direct endorsement of transformation choice"
            rationale = "Sources establish measurement context, not that no transformation is optimal. Current native-scale analysis is a declared project decision, subject to model diagnostics."
        if parameter in {"Pupil change rate", "Eyelid aperture", "Distance from center", "Gaze centrality", "Gaze velocity", "Gaze dispersion", "Fixation rate"}:
            rationale += " Exact exported derived metric and transformation are not specifically validated by these references."
        if column in NOT_IN_STEP5:
            status = "Not a fitted outcome; measurement context only"
        rows.append({"col_name": column, "Parameter": parameter, "Transformation": transformation,
                     "Number of papers": len(keys), "Papers": ", ".join(numbers[k] for k in keys)})
        evidence.append({"Column": column, "transformation_in_models": step5_text(column, transformation),
            "paper_citations": " | ".join(REFERENCES[k]["citation"] + " DOI: " + REFERENCES[k]["doi"] for k in keys),
            "paper_files": " | ".join(str(paper_paths[k]) for k in keys),
            "paper_dois": " | ".join(REFERENCES[k]["doi"] for k in keys),
            "evidence_status": status, "transformation_rationale": rationale,
            "evidence_details": " | ".join(numbers[k] + " PDF p. " + REFERENCES[k]["pages"] + ": " + REFERENCES[k]["evidence"] for k in keys),
            "reference_count_meaning": "Verified full-text contextual/method references; not a vote count for the selected transformation"})
    return pd.DataFrame(rows), pd.DataFrame(evidence)


def transformation_policy(column, series):
    if column in COL.values():
        parameter = next(p for p, c in COL.items() if c == column)
        transformation = USED[parameter][0][0]
        return "Physiological outcome", STEP5_TEXT[transformation], "Existing literature mapping; transform before model-specific standardization"
    if column in METADATA_FIELDS or column.endswith("_Date") or column.startswith("ucm_GPS_") or column == "ucm_MAG_hdg":
        return "Metadata", "Not applicable", "Identifier, timestamp or positioning metadata; not a continuous exposure transformation"
    if column in QC_FIELDS or column.startswith("empatica_hrv_fd_5min_") or column.startswith("SVF_") and column != "SVF":
        return "QC / processing metadata", "Not applicable", "Availability, quality or processing descriptor; retained for checking"
    if column == "unidentify_prop":
        return "Classification diagnostic", "Not applicable to exposure model", "Unidentified sound share; excluded before sound PCA"
    if column == "category" or column.endswith(("_SR", "_lux_cat", "__lux_cat")) or column == "ucm_IO_flag":
        return "Categorical / ordinal code", "No continuous transformation", "Use declared category coding; numerical storage does not imply a continuous measurement"
    if column.startswith("questionnaire_"):
        score = any(x in column for x in ["_mean_", "_comfort_", "_pleasantness_", "_familiarity_"])
        return ("Questionnaire score" if score else "Questionnaire / participant metadata", "Native score or categorical coding; no automatic log", "Repeated questionnaire values do not become independent ten-second observations; model depends on item/score type")
    if column.endswith("_%") or column in {"SVF", "gvi_50m"} or column.startswith(("gvi_", "tree_phase_", "grass_phase_", "plant_phase_")):
        return "SVF / greenery exposure", "Native scale before model-specific centering/scaling", "Bounded exposure; no automatic log/logit. Boundary values and model role must be checked"
    if column == "ucm_SND_dBA" or column.startswith(SOUND_PROPORTION_PREFIXES) or column.endswith(("_score", "_prop")) or column in {"dominant_frequency_hz", "spectral_centroid_hz", "spectral_rolloff85_hz"}:
        return "Sound exposure", "Native scale; within-bout centering and unit-SD scaling for sound PCA", "dBA is already logarithmic; do not log again. Proportions/scores are not probabilities of equivalent meaning; current PCA is correlation-based"
    if pd.api.types.is_numeric_dtype(series):
        return "Numeric measurement / derived field", "Native scale before model-specific centering/scaling", "Native scale; within-bout centering and pooled-SD scaling in H4 when entered"
    return "Text / categorical field", "No continuous transformation", "Preserve labels; no z-score or numeric log of text"

def make_column_inventory(physio_counts):
    existing = physio_counts.set_index("col_name")
    rows = []
    for column in normalization_columns:
        values = core[column]
        role, treatment, basis = transformation_policy(column, values)
        numeric = pd.to_numeric(values, errors="coerce") if pd.api.types.is_numeric_dtype(values) else pd.Series(dtype=float)
        valid = numeric.dropna()
        ref = existing.loc[column].to_dict() if column in existing.index else {}
        rows.append({"col_name": column, "Parameter": ref.get("Parameter", column),
            "Role": role, "Transformation": treatment, "Basis": basis,
            "Scaling": "Model-specific; metadata/QC/categorical fields are not automatically standardized",
            "Number of papers": ref.get("Number of papers", 0), "Papers": ref.get("Papers", ""),
            "Origin": "Bilateral mean" if column in AVERAGES else "Merged source",
            "Core rows": len(core), "Missing": int(values.isna().sum()), "Unique": values.nunique(),
            "Nonpositive": int((valid <= 0).sum()) if len(valid) else None,
            "Minimum": valid.min(), "Maximum": valid.max(),
            "Skewness": valid.skew() if len(valid) >= 3 and valid.nunique() > 1 else None,
            "Nonfinite": int((~np.isfinite(numeric.dropna())).sum()) if len(numeric) else 0})
    table = pd.DataFrame(rows)
    assert not table.col_name.duplicated().any()
    assert set(table.col_name) == set(normalization_columns)
    return table

print("functions defined (normalization tables + column inventory)")


functions defined (normalization tables + column inventory)


In [4]:
def write_csv(path, frame):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=OUT_DIR, suffix='.tmp', delete=False) as handle:
        tmp = Path(handle.name)
        frame.to_csv(handle, index=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)
    print(path.resolve())

OUT_ALL = OUT_DIR / 'Normalization_H1toH4_all_parameters.csv'
OUT_DIAG = OUT_DIR / 'Normalization_H1toH4_diagnostics.csv'

physio_counts, phys = make_normalization_tables()
count = make_column_inventory(physio_counts)
all_params = count.rename(columns={
    'col_name': 'column_name',
    'Parameter': 'parameter_label',
    'Role': 'role',
    'Transformation': 'transformation_in_models_default',
    'Number of papers': 'number_of_papers',
    'Papers': 'paper_keys',
}).merge(phys, left_on='column_name', right_on='Column', how='left')
all_params['transformation_in_models'] = all_params['transformation_in_models'].fillna(all_params['transformation_in_models_default'])
all_params['paper_citations'] = all_params['paper_citations'].fillna('')
all_params = all_params[all_params['column_name'].map(in_normalization_scope)]
all_params['evidence_status'] = all_params['evidence_status'].fillna('No literature claim: metadata, QC or unmodelled derived field')
all_params = all_params[[
    'column_name', 'parameter_label', 'role', 'transformation_in_models',
    'number_of_papers', 'paper_keys', 'paper_citations',
    'Missing', 'Nonpositive', 'Skewness',
    'paper_files', 'paper_dois', 'evidence_status', 'transformation_rationale',
    'evidence_details', 'reference_count_meaning',
]].sort_values(['role', 'column_name'])
diag = count[count['col_name'].map(in_normalization_scope)][['col_name', 'Role', 'Missing', 'Nonpositive', 'Minimum', 'Maximum', 'Skewness', 'Nonfinite']].rename(columns={'col_name': 'column_name'})
write_csv(OUT_ALL, all_params)
write_csv(OUT_DIAG, diag)
print('all_parameters=%d with_papers=%d' % (len(all_params), int((all_params['number_of_papers'].fillna(0) > 0).sum())))


C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_diagnostics.csv
all_parameters=41 with_papers=27
